# Block 5 — PyTorch Part 1: Tensors, Neural Nets, Backprop

> **The big idea.** In Block 3 you trained a *linear* sklearn model to predict customer churn.
> Today you swap it for a model class that can **bend** the decision boundary — a neural
> network — and learn the three primitives that make every modern deep learning system work:
> **tensors** (the data), the **forward pass** (how a network turns inputs into predictions),
> and the **backward pass** (how it learns from its mistakes).

**How this notebook works**

- Short explanations, then small hands-on tasks marked **🎯** for you to complete.
- Every task is followed by a quick **self-check** so you know it worked.
- Whenever you need a specific function, we tell you which one. 🙂
- Runs top-to-bottom on **Google Colab** — `Runtime → Run all`.

**What you will build**

1. A **tensor warm-up** — recreating Block 1's NumPy operations in PyTorch.
2. The **XOR / n-bit parity** problem — proving by counter-example that a single linear unit
   cannot learn even the simplest non-linear task, then watching one hidden layer solve it.
3. A **universal-approximation** demo — a tiny one-hidden-layer net fitting a noisy sine,
   improving as you add hidden units.
4. A **churn MLP** — the neural counterpart of your Block 3 logistic regression on the TELCO
   dataset, compared head to head against that sklearn baseline.


## 0. Setup

You only need a handful of libraries today:

- **`torch`** — the tensor library and its **autograd** engine.
- **`torch.nn`** — ready-made layers (`Linear`, activations, …) and losses (`BCEWithLogitsLoss`, `MSELoss`).
- **`torch.optim`** — gradient-descent optimizers (`SGD`, `Adam`).
- **`numpy`**, **`matplotlib`**, **`sklearn`** — for data prep, plotting, and the classical baseline.


In [ ]:
!pip install -q torch numpy matplotlib scikit-learn tqdm

In [ ]:
import os, shutil

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE0-public"          # public repo — students need NO token
DATA_FILE  = "customer_churn_TELCO.csv"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab() and not os.path.exists(DATA_FILE):
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

    # Find the CSV inside the cloned repo and copy ONLY that file to the working dir.
    _src = None
    for _dir, _, _files in os.walk(REPO_NAME):
        if DATA_FILE in _files:
            _src = os.path.join(_dir, DATA_FILE)
            break
    if _src is None:
        raise FileNotFoundError(
            f"Could not find {DATA_FILE} in {REPO_OWNER}/{REPO_NAME}.")
    shutil.copy(_src, DATA_FILE)

# Resolve the dataset path for both environments:
#   - Colab: the CSV was copied to the working dir above.
#   - Local clone: the CSV lives under data/ at the repo root (./data, ../data, or ../../data).
DATA_PATH = DATA_FILE
if not os.path.exists(DATA_PATH):
    for _cand in (os.path.join("data", DATA_FILE),
                  os.path.join("..", "data", DATA_FILE),
                  os.path.join("..", "..", "data", DATA_FILE)):
        if os.path.exists(_cand):
            DATA_PATH = _cand
            break
print("Dataset available at:", os.path.abspath(DATA_PATH))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("torch", torch.__version__)

## 1. Tensor warm-up

A **PyTorch tensor** is, for our purposes, the same object you used in Block 1 — an
$n$-dimensional grid of numbers, shape $(d_1, d_2, \dots, d_n)$ — with two extra powers:

1. **It can live on a GPU.** Move it with `.to("cuda")` and every operation runs on the GPU.
2. **It tracks the chain of operations applied to it**, so PyTorch can compute gradients
   automatically (this is called **autograd**, and you'll see it at the end of this section).

Practically, the NumPy reflexes you built in Block 1 transfer almost verbatim. The cheat sheet:

| NumPy | PyTorch | Notes |
|---|---|---|
| `np.array([...])` | `torch.tensor([...])` | `torch` defaults to `float32`, NumPy to `float64`. |
| `np.zeros((2,3))` | `torch.zeros(2, 3)` | Same shape semantics. |
| `np.arange`, `np.linspace` | `torch.arange`, `torch.linspace` | Identical. |
| `array.shape`, `array.ndim` | `tensor.shape`, `tensor.ndim` | Identical. |
| `A @ B` | `A @ B` | Same operator. |
| `np.sum(A, axis=0)` | `A.sum(dim=0)` | PyTorch uses `dim=`, not `axis=`. |


### 🎯 Task 1.1 — Creating tensors

Recreate a handful of Block-1 constructors. Build a tensor `a` holding the matrix
$\bigl[\begin{smallmatrix}1&2&3\\4&5&6\end{smallmatrix}\bigr]$ in `float32`, a 2×3 zeros tensor, a 2×3 ones tensor, an arange from 0 to 10 stepping by 2, a 5-point linspace from 0 to 1, and a 2×3 standard-normal sample.

In [ ]:
# 🎯 TODO — create the tensors below
a = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
ar = torch.arange(0, 10, 2)
lin = torch.linspace(0, 1, 5)
rnd = torch.randn(2, 3)
# end TODO

print("a:\n", a)
print("zeros:\n", zeros)
print("arange :", ar)
print("linspace:", lin)
print("randn shape:", rnd.shape)

# self-checks
assert a.dtype == torch.float32
assert zeros.shape == (2, 3) and ones.shape == (2, 3)
assert ar.tolist() == [0, 2, 4, 6, 8]
assert lin.shape == (5,)
assert rnd.shape == (2, 3)

### 🎯 Task 1.2 — Shape inspection

Every time you touch an unfamiliar tensor, ask four questions: **shape, ndim, numel, dtype**. Print them for `a`.

In [ ]:
# 🎯 TODO — print the four shape attributes of `a`
print("shape:", a.shape)
print("ndim :", a.ndim)
print("numel:", a.numel())
print("dtype:", a.dtype)
# end TODO

assert a.shape == (2, 3)
assert a.ndim == 2
assert a.numel() == 6

### 🎯 Task 1.3 — Indexing and slicing

PyTorch indexing is the same as NumPy: a single integer picks a row, `:` selects all, and a
boolean mask keeps positions where the mask is `True`. Build the 3×4 matrix
`M = torch.arange(12).reshape(3, 4)` and access:

- `M[0, 0]` — single element,
- `M[1]` — the second row,
- `M[:, 2]` — the third column,
- `M[M > 5]` — every element greater than 5.

In [ ]:
# 🎯 TODO — build M and try the four index forms
M = torch.arange(12).reshape(3, 4)
print("M:\n", M)
print("M[0, 0]   =", M[0, 0].item())
print("M[1]      =", M[1])
print("M[:, 2]   =", M[:, 2])
print("M[M > 5]  =", M[M > 5])
# end TODO

assert M[0, 0].item() == 0
assert M[:, 2].tolist() == [2, 6, 10]
assert M[M > 5].tolist() == [6, 7, 8, 9, 10, 11]

### 🎯 Task 1.4 — Broadcasting

**Broadcasting rule.** When you combine two tensors elementwise, PyTorch aligns their shapes
**right-to-left**. Two dimensions are compatible if they are equal or one of them is $1$.
The smaller tensor is then *virtually* stretched along the size-$1$ dimensions to match.

Worked example: a row of shape $(3,)$ added to a column of shape $(3, 1)$ broadcasts to $(3, 3)$:

$$\begin{pmatrix}1 & 2 & 3\end{pmatrix} + \begin{pmatrix}10 \\ 20 \\ 30\end{pmatrix}
= \begin{pmatrix}11 & 12 & 13 \\ 21 & 22 & 23 \\ 31 & 32 & 33\end{pmatrix}.$$

Reproduce this in code.

In [ ]:
row = torch.tensor([1.0, 2.0, 3.0])           # shape (3,)
col = torch.tensor([[10.0], [20.0], [30.0]])  # shape (3, 1)

print("row.shape:", row.shape)
print("col.shape:", col.shape)

# 🎯 TODO — add row and col, then print the broadcast result
out = row + col
# end TODO

print("(row + col).shape:", out.shape)
print(out)

assert out.shape == (3, 3)
assert torch.allclose(out[0], torch.tensor([11.0, 12.0, 13.0]))

### 🎯 Task 1.5 — Reductions

Reductions collapse a tensor along an axis: sum, mean, max, …. In PyTorch the axis is
called **`dim`** (NumPy calls it `axis`). The flag `keepdim=True` preserves the rank of the
output by keeping a size-$1$ dimension where the reduction happened — this is what makes the
result broadcast cleanly back against the original.

Build a 3×4 matrix `X = torch.arange(12, dtype=torch.float32).reshape(3, 4)` and compute its
total sum, its column-wise sum (`dim=0`), its row-wise sum with `keepdim=True`, and its
column-wise mean.

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("X:\n", X)

# 🎯 TODO — compute the four reductions
total = X.sum()
col_sum = X.sum(dim=0)
row_sum_kd = X.sum(dim=1, keepdim=True)
col_mean = X.mean(dim=0)
# end TODO

print("X.sum()                   =", total.item())
print("X.sum(dim=0)              =", col_sum)
print("X.sum(dim=1, keepdim=True):\n", row_sum_kd)
print("X.mean(dim=0)             =", col_mean)

assert col_sum.shape == (4,)
assert row_sum_kd.shape == (3, 1)

### 🎯 Task 1.6 — Matmul vs. elementwise

Two operations that look similar but mean very different things:

- **Matrix multiplication.** $C = A B$ with $C_{ij} = \sum_k A_{ik} B_{kj}$.
  Shape rule: $(m, k) \times (k, n) \to (m, n)$. In code: `A @ B` or `torch.matmul(A, B)`.
- **Elementwise (Hadamard) product.** $C_{ij} = A_{ij} \cdot B_{ij}$. Requires equal (or
  broadcastable) shapes. In code: `A * B`.

Try both.

In [ ]:
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = torch.randn(3, 4)

# 🎯 TODO — compute the matmul A @ B and the elementwise A * C
prod_at = A @ B
prod_matmul = torch.matmul(A, B)
elem = A * C
# end TODO

print("A @ B shape       :", prod_at.shape)
print("torch.matmul shape:", prod_matmul.shape)
print("A * C (elementwise) shape:", elem.shape)

assert torch.allclose(prod_at, prod_matmul)
assert prod_at.shape == (3, 2)
assert elem.shape == (3, 4)

### 🎯 Task 1.7 — Reshape, view, transpose

Three ways to rearrange a tensor without changing its values:

- **`reshape(*new_shape)`** — may copy if needed.
- **`view(*new_shape)`** — same as `reshape` but requires contiguous memory; shares storage.
- **`T` / `transpose(dim0, dim1)`** — swaps two axes. Shape `(m, n) → (n, m)`.

Build the vector `v = torch.arange(12)` and reshape / view it; then take the transpose of `M`.

In [ ]:
v = torch.arange(12)

# 🎯 TODO — produce three reshaped/transposed views
v_3x4 = v.reshape(3, 4)
v_2x6 = v.view(2, 6)
M_T = M.T
# end TODO

print("v.reshape(3, 4):\n", v_3x4)
print("v.view(2, 6)   :\n", v_2x6)
print("M.T:\n", M_T)

assert v_3x4.shape == (3, 4)
assert v_2x6.shape == (2, 6)
assert M_T.shape == (M.shape[1], M.shape[0])

### 🎯 Task 1.8 — Gradient teaser

This is what makes a PyTorch tensor more than a NumPy array. Set `requires_grad=True` on
a leaf tensor, do any differentiable computation, then call `.backward()` on the result. The
gradient is deposited in `x.grad`.

Try $y = x^3$ at $x = 2$. The analytical derivative is $\dfrac{dy}{dx} = 3x^2$, so at $x=2$
we expect $12$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# 🎯 TODO — compute y = x^3 and call backward()
y = x ** 3
y.backward()
# end TODO

print("x =", x.item(), "  y = x**3 =", y.item(), "  dy/dx =", x.grad.item())
# Analytical: dy/dx = 3 * x**2 = 12.0
assert torch.isclose(x.grad, torch.tensor(12.0))

## 2. From tensors to a trainable model

So far we have treated tensors as data containers and seen that they can track gradients. To actually train something, we need to assemble tensors into a model, define how it maps inputs to outputs, and set up the loop that updates its parameters. This section introduces the building blocks that every example afterwards (XOR, parity, the sine fit, and churn) will reuse: how to define a model, why we need nonlinear activations, the standard training loop, and how parameters are initialized. Once these are in place, each later task is just an application of the same pattern.

### 2.1 Building a model with `nn.Module`

Every model in PyTorch is a subclass of `torch.nn.Module`. You define two things:

- **`__init__`**: register the layers and any learnable parameters. Anything you assign as `self.something = nn.Linear(...)` is tracked automatically, so its weights appear in `model.parameters()` and receive gradients during `.backward()`. Always call `super().__init__()` first, since that is what sets up the parameter tracking.
- **`forward`**: define how an input tensor flows through the layers to produce an output. This is just the forward pass written as plain tensor operations.

A linear layer `nn.Linear(in, out)` computes $y = xW^\top + b$, where $W$ has shape `(out, in)` and $b$ has shape `(out,)`. Stacking linear layers with a nonlinearity between them is what lets the model represent functions that a single linear map cannot.

You never call `forward` directly. Calling `model(x)` runs it through `__call__`, which also performs the bookkeeping PyTorch needs, such as hooks and autograd tracking.

Once a model is built, you can inspect what got registered with `model.parameters()` or `model.state_dict()`.

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()                          # sets up parameter tracking
        self.fc1 = nn.Linear(in_dim, hidden_dim)    # first linear layer
        self.act = nn.ReLU()                        # nonlinear activation
        self.fc2 = nn.Linear(hidden_dim, out_dim)   # output layer

    def forward(self, x):
        x = self.act(self.fc1(x))                   # forward pass, layer by layer
        return self.fc2(x)

# instantiate a small network: 2 inputs, 8 hidden units, 1 output
model = MLP(in_dim=2, hidden_dim=8, out_dim=1)

# inspect what got registered
print(model)
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

### 🎯 2.2 Activation functions

A network made only of linear layers is still linear: composing $x \mapsto W_2(W_1 x + b_1) + b_2$ just gives another affine map $x \mapsto W x + b$, no matter how many layers you stack. To represent nonlinear functions, we insert a nonlinear activation between layers. This is the single ingredient that makes depth meaningful, and it is exactly what lets the MLP solve XOR while a linear model cannot.

Three common choices:

- **Sigmoid**: $\sigma(x) = \dfrac{1}{1 + e^{-x}}$, squashes inputs into $(0, 1)$. Useful for probabilities at the output, but its derivative is at most $0.25$, so stacking many sigmoids shrinks gradients geometrically toward zero.
- **Tanh**: $\tanh(x)$, squashes into $(-1, 1)$ and is zero-centered, which often trains better than sigmoid in hidden layers, though it still saturates for large $|x|$.
- **ReLU**: $\mathrm{ReLU}(x) = \max(0, x)$. Cheap to compute and does not saturate for positive inputs, so it largely avoids the shrinking-gradient problem. It is the default choice for hidden layers in most modern networks.

The pattern "linear layer, then activation, then linear layer" is what we used in `forward` above, and it is the basic unit we reuse throughout the rest of the notebook.

**Task:** In the cell below, implement the three activation functions in the `🎯 TODO` blocks, then run the cell to plot them over the range $[-5, 5]$. Look at the edges of each plot: sigmoid and tanh flatten out (saturation), while ReLU stays linear for positive inputs. This is the visual signature of the gradient behavior described above.

In [ ]:
# input range to evaluate the activations on
x = torch.linspace(-5, 5, 200)

def sigmoid(x):
    # 🎯 TODO — implement sigmoid: 1 / (1 + exp(-x))
    return 1 / (1 + torch.exp(-x))
    # end TODO

def tanh(x):
    # 🎯 TODO — implement tanh (you may use torch.tanh)
    return torch.tanh(x)
    # end TODO

def relu(x):
    # 🎯 TODO — implement relu: max(0, x) elementwise
    return torch.clamp(x, min=0)
    # end TODO

# plot each activation
activations = {"sigmoid": sigmoid, "tanh": tanh, "relu": relu}

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, fn) in zip(axes, activations.items()):
    ax.plot(x.numpy(), fn(x).numpy())
    ax.set_title(name)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. XOR and n-bit parity

We now ask a simple question: **how powerful is a single linear layer?** The answer ("not very")
is what motivates the rest of deep learning.

**Linear units decide with hyperplanes.** A single artificial neuron with sigmoid output
computes
$$\hat{y} = \sigma(\mathbf{w}^\top \mathbf{x} + b), \qquad \sigma(z) = \frac{1}{1 + e^{-z}},$$
and its **decision boundary** (the set of inputs where it switches from predicting class 0 to
class 1) is the hyperplane $\mathbf{w}^\top \mathbf{x} + b = 0$. In $\mathbb{R}^2$ that's a
straight line.

**XOR is not linearly separable.** The XOR function maps
$(0,0)\to 0,\; (0,1)\to 1,\; (1,0)\to 1,\; (1,1)\to 0$. Plot the four points and try to
draw a straight line with the two "1"s on one side and the two "0"s on the other — you can't.
**Sketch of the impossibility argument:** suppose such $(\mathbf{w}, b)$ existed. Then
$\mathbf{w}^\top(0,1)^\top + b > 0$ and $\mathbf{w}^\top(1,0)^\top + b > 0$ give
$w_1 + w_2 + 2b > 0$. But $\mathbf{w}^\top(0,0)^\top + b < 0$ and
$\mathbf{w}^\top(1,1)^\top + b < 0$ give $w_1 + w_2 + 2b < 0$. Contradiction. So **at most three**
of the four points can be classified correctly by *any* linear unit.

**One hidden layer is enough.** Compose a linear map with a non-linearity, exactly the
"linear, then activation, then linear" pattern from section 2:
$$\mathbf{h} = \phi(W_1 \mathbf{x} + \mathbf{b}_1), \qquad \hat{y} = \sigma(W_2 \mathbf{h} + b_2),$$
with $\phi$ any non-polynomial activation (we'll use $\tanh$, one of the activations from
section 2.2). With just two hidden units you can already realise
$$x_1 \oplus x_2 = (x_1 \vee x_2) \wedge \neg(x_1 \wedge x_2),$$
and you'll see the network discover exactly this below.

### 3.1 The XOR truth table

In [ ]:
X_xor = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float32)
y_xor = torch.tensor([0, 1, 1, 0], dtype=torch.float32).unsqueeze(1)  # shape (4, 1) to match model output
print("X_xor:\n", X_xor)
print("y_xor:", y_xor.squeeze().tolist())

### 3.2 The training pipeline: gradient descent, BCE, autograd

Three pieces will appear in every PyTorch training loop you write today:

1. **Gradient descent.** With parameters $\theta$ and loss $\mathcal{L}(\theta)$, the update is
   $$\theta_{t+1} = \theta_t - \eta\, \nabla_\theta \mathcal{L}(\theta_t),$$
   where $\eta$ is the **learning rate**. `torch.optim.SGD` does exactly this.

2. **Binary cross-entropy loss.** For a target $y \in \{0, 1\}$ and a predicted probability
   $\hat{p} = \sigma(z)$ where $z$ is the network's raw output (the **logit**),
   $$\mathcal{L}_{\text{BCE}}(y, \hat{p}) = -\bigl[\, y \log \hat{p} + (1 - y) \log(1 - \hat{p}) \,\bigr].$$
   `nn.BCEWithLogitsLoss` takes the **logit** $z$ directly and applies $\sigma$ internally, which
   is more numerically stable than computing $\sigma$ first and then $\log$ (a log-sum-exp trick).

3. **Forward and backward pass.** The **forward pass** composes layers to produce $z$ and the
   loss. The **backward pass** is one call, `loss.backward()`, that runs autograd in reverse
   over every operation you did, applying the chain rule
   $$\frac{\partial \mathcal{L}}{\partial \theta} = \frac{\partial \mathcal{L}}{\partial z}
     \cdot \frac{\partial z}{\partial \theta}$$
   to fill the `.grad` attribute of every parameter. The optimizer then reads those gradients
   and takes a step. (We'll derive the chain-rule expressions by hand in the next block, for now
   autograd handles it.)

Put together, every training step is the same five lines, in this order:

1. `optimizer.zero_grad()` — clear last step's gradients
2. `logits = model(X)` — forward pass
3. `loss = loss_fn(logits, y)` — evaluate the loss
4. `loss.backward()` — backward pass, fills `.grad`
5. `optimizer.step()` — update parameters

Step 1 is not optional: `.backward()` *accumulates* into `.grad` rather than overwriting it
(because a parameter can feed several downstream paths, and their gradients must sum). If you
forget `zero_grad`, you add this step's gradient to the last one's and the updates are wrong.
This is the single most common beginner bug, so it is worth memorizing now.

The training helper below packages exactly these five steps into the loop you'll use for every
model in this section.

In [ ]:
def train(model, X, y, loss_fn, epochs=2000, lr=0.1):
    """Standard full-batch training loop. Returns the loss history.

    Each iteration is the same five steps, in the canonical order:
      1. zero_grad  2. forward  3. loss  4. backward  5. step
    """
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for _ in tqdm(range(epochs)):
        optimizer.zero_grad()    # 1. clear gradients from previous step
        logits = model(X)        # 2. forward pass
        loss = loss_fn(logits, y)  # 3. evaluate the loss
        loss.backward()          # 4. autograd populates every .grad
        optimizer.step()         # 5. theta <- theta - lr * grad
        history.append(loss.item())
    return history

bce = nn.BCEWithLogitsLoss()

### 🎯 Task 3.1 — Try the linear model on XOR

Fit a single-layer linear model (equivalent to logistic regression) to XOR using the `train`
helper above. The impossibility argument from section 3 predicts it can get at most three of the
four points right, so watch the loss plateau well above zero and check which point it misclassifies.

In [ ]:
torch.manual_seed(0)

# 🎯 TODO — build a single nn.Linear(2, 1) and train it on (X_xor, y_xor)
linear = nn.Linear(2, 1)    # one linear unit: decision boundary is a single line
hist_linear = train(linear, X_xor, y_xor, bce, epochs=2000, lr=0.1)
# end TODO

# evaluate under no_grad: inference only, no need to track gradients
with torch.no_grad():
    preds_linear = (torch.sigmoid(linear(X_xor)) > 0.5).float()
acc_linear = (preds_linear == y_xor).float().mean().item()
print("\nlinear predictions:", preds_linear.squeeze().tolist(),
      " targets:", y_xor.squeeze().tolist())
print("linear accuracy   :", acc_linear)

# self-check: no linear unit can do better than 3 out of 4 (the impossibility argument from section 3)
assert acc_linear <= 0.75, "a linear unit should classify at most 3 of the 4 XOR points"

The linear model **caps at 75 %** — it can match three of the four points but never all
four. There is no hyperplane in $\mathbb{R}^2$ that does better, no matter how long you train.

### 🎯 Task 3.2 — Add one hidden layer

Now stack two `Linear` layers with a $\tanh$ non-linearity between them, exactly the model
pattern from section 2.1. Use 4 hidden units and train with the same `train` helper. Two units
are enough in principle (section 3), but a few extra make training reliably converge from a
random start. Watch the same data that defeated the linear model become trivial: the loss should
fall close to zero and all four points should be classified correctly.

In [ ]:
torch.manual_seed(0)

# 🎯 TODO — define the MLP: Linear(2, 4) -> Tanh -> Linear(4, 1)
# nn.Sequential is shorthand for the section 2.1 pattern: it chains modules and its
# forward() just passes the input through each in order. Tanh is the activation from 2.2.
mlp = nn.Sequential(
    nn.Linear(2, 4),
    nn.Tanh(),
    nn.Linear(4, 1),
)
hist_mlp = train(mlp, X_xor, y_xor, bce, epochs=2000, lr=0.1)
# end TODO

# same evaluation as the linear model, so the two accuracies are directly comparable
with torch.no_grad():
    preds_mlp = (torch.sigmoid(mlp(X_xor)) > 0.5).float()
acc_mlp = (preds_mlp == y_xor).float().mean().item()
print("\nmlp predictions:", preds_mlp.squeeze().tolist(),
      " targets:", y_xor.squeeze().tolist())
print("mlp accuracy   :", acc_mlp)

# self-check: one hidden layer with a non-linearity solves XOR exactly
assert acc_mlp == 1.0, "the MLP should classify all 4 XOR points correctly"

One hidden layer is **enough**: accuracy jumps to 100 %. The hidden layer first maps the four
points into a new space where they *are* linearly separable, and the output unit then splits
them with a single hyperplane. Seen back in the original input space, that boundary looks
curved, and the four XOR points fall into the right regions neatly.

In [ ]:
plt.plot(hist_linear, label="linear")
plt.plot(hist_mlp, label="MLP (1 hidden layer)")
plt.title("XOR loss curves"); plt.xlabel("epoch"); plt.ylabel("BCE")
plt.legend(); plt.show()

# the linear curve flattens at a positive BCE (it cannot separate XOR),
# while the MLP drives the loss toward zero
print(f"final linear BCE: {hist_linear[-1]:.4f}")
print(f"final MLP BCE   : {hist_mlp[-1]:.4f}")

### 🎯 Task 3.3 — Generalize to n-bit parity

**Parity** is XOR generalized to $n$ bits:
$$y(\mathbf{x}) = \bigoplus_{i=1}^{n} x_i, \qquad \mathbf{x} \in \{0,1\}^n.$$
The "1" regions are the $2^{n-1}$ corners of the hypercube with an odd number of 1-bits, and the
"0" regions are the remaining $2^{n-1}$ corners. The two sets are interleaved on the hypercube
(every corner's neighbors flip parity), so the problem becomes geometrically harder as $n$ grows.
This is a classic example of why **width and depth** matter as data become more entangled: a
one-hidden-layer network can still represent parity, but it needs more hidden units as $n$ grows,
and training gets harder well before that limit.

Fill in the `make_parity` helper, then train a small MLP for $n \in \{2, 3, 4, 5\}$ and watch how
accuracy and the number of epochs needed change with $n$.

In [ ]:
def make_parity(n):
    """Return all 2**n bitstrings of length n and their XOR parity."""
    # 🎯 TODO — build the bit-matrix and compute the parity column
    bits = torch.tensor(
        [[(i >> k) & 1 for k in range(n)] for i in range(2 ** n)],
        dtype=torch.float32,
    )
    # parity = sum of bits mod 2: 1 if an odd number of bits are set, else 0
    parity = bits.sum(dim=1).remainder(2).unsqueeze(1)
    # end TODO
    return bits, parity

parity_accs = {}
for n in [2, 3, 4, 5]:
    X_n, y_n = make_parity(n)
    torch.manual_seed(0)
    # hidden width scales with n (4*n), since wider layers are needed as parity grows
    net = nn.Sequential(
        nn.Linear(n, 4 * n),
        nn.Tanh(),
        nn.Linear(4 * n, 1),
    )
    train(net, X_n, y_n, bce, epochs=3000, lr=0.1)
    with torch.no_grad():
        acc = ((torch.sigmoid(net(X_n)) > 0.5).float() == y_n).float().mean().item()
    parity_accs[n] = acc
    print(f"n = {n}   accuracy = {acc:.2f}")

# self-check: small parities are easy; the larger ones are visibly harder for this small net
for k in (2, 3):
    assert parity_accs[k] == 1.0, f"{k}-bit parity should be solved exactly by this net"
assert parity_accs[4] >= 0.75, "4-bit parity should be mostly solved, but may not be perfect"

## 4. Universal approximation

XOR proved a hidden layer is *sometimes* enough. A much stronger statement is true:

> **Universal Approximation Theorem** (Cybenko 1989, Hornik 1991). Let $\phi$ be any continuous,
> non-polynomial activation function. Then for every continuous $f : [0, 1]^d \to \mathbb{R}$
> and every $\varepsilon > 0$, there exist $H \in \mathbb{N}$, weights
> $\{\mathbf{w}_i\}_{i=1}^{H} \subset \mathbb{R}^d$, $\{v_i\}_{i=1}^{H} \subset \mathbb{R}$, and
> biases $\{b_i\}_{i=1}^{H} \subset \mathbb{R}$ such that
> $$\Bigl| f(\mathbf{x}) - \sum_{i=1}^{H} v_i\, \phi(\mathbf{w}_i^\top \mathbf{x} + b_i) \Bigr|
>   < \varepsilon \quad \forall \mathbf{x} \in [0, 1]^d.$$

In English: **a single hidden layer of sufficient width can approximate any continuous function
arbitrarily well.**

Two caveats are worth shouting:

1. This is an **existence** result. It says the required weights *exist*, not that they are easy
   to find with gradient descent, not that $H$ is small, not that you have enough data. This is
   the same gap you already saw with parity: representable in principle is not the same as
   trainable in practice.
2. In practice we often prefer **depth** to extreme width, since deeper networks can express the
   same functions with far fewer parameters.

You'll see the first point concretely below: we fit a one-hidden-layer network to a nonlinear
target, widen it, and watch the approximation improve while training stays finicky.

### 4.1 A noisy sine target

We'll fit
$$f(x) = \sin(x) + \varepsilon, \qquad x \in [-2\pi, 2\pi], \quad \varepsilon \sim \mathcal{N}(0, 0.2^2)$$
with a one-hidden-layer net. This is a regression task (continuous target) rather than
classification, so we swap the BCE loss used for XOR and parity for the **mean squared error**
$$\mathcal{L}_{\text{MSE}} = \frac{1}{N}\sum_{i=1}^{N} (y_i - \hat{y}_i)^2.$$
The added noise $\varepsilon$ is what makes the fit interesting: the network should recover the
underlying $\sin(x)$, not chase individual noisy points, which is exactly the overfitting tension
we look at next.

In [ ]:
torch.manual_seed(0)
x_reg = torch.linspace(-2 * np.pi, 2 * np.pi, 200).unsqueeze(1)   # shape (200, 1): column vector for the net
y_reg = torch.sin(x_reg) + 0.2 * torch.randn_like(x_reg)         # clean signal plus Gaussian noise

plt.scatter(x_reg.numpy(), y_reg.numpy(), s=8, color="#888", label="data")
plt.plot(x_reg.numpy(), torch.sin(x_reg).numpy(), color="#2b2d6b", label="sin(x)")
plt.title("Regression target"); plt.legend(); plt.show()

### 🎯 Task 4.1 — Sweep the hidden width

Define a factory `make_net(H)` that returns a one-hidden-layer net of width $H$ with a $\tanh$
activation (the same `nn.Sequential` pattern as before, now with the width left as an argument),
and train it on the noisy sine target with the `train` helper for several values of $H$. Watch
the fit tighten as $H$ grows: this is the universal approximation theorem made visible, with
larger $H$ buying a closer approximation. Note also how many epochs each width needs, since wider
is not automatically easier to train.

In [ ]:
def make_net(hidden_units):
    # 🎯 TODO — return nn.Sequential(Linear(1, H), Tanh, Linear(H, 1))
    return nn.Sequential(
        nn.Linear(1, hidden_units),
        nn.Tanh(),
        nn.Linear(hidden_units, 1),
    )
    # end TODO

mse = nn.MSELoss()        # regression loss, swapped in for BCE (see 4.1)
widths = [2, 4, 8, 32, 128]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(x_reg.numpy(), y_reg.numpy(), s=8, color="#bbb", label="data")
for H in widths:
    torch.manual_seed(0)    # same init seed per width, so differences come from H alone
    net = make_net(H)
    train(net, x_reg, y_reg, mse, epochs=3000, lr=0.01)
    with torch.no_grad():
        y_hat = net(x_reg)
    ax.plot(x_reg.numpy(), y_hat.numpy(), label=f"H = {H}")
ax.set_title("one-hidden-layer net: approximation vs. width")
ax.legend(); plt.show()

# self-check: make_net builds a 3-module Sequential of the expected shape
_probe = make_net(8)
assert len(_probe) == 3, "expected exactly 3 modules: Linear, Tanh, Linear"
assert isinstance(_probe[0], nn.Linear) and isinstance(_probe[1], nn.Tanh) and isinstance(_probe[2], nn.Linear)
assert _probe[0].out_features == 8 and _probe[2].in_features == 8, "hidden width should be 8"

As $H$ grows $2 \to 4 \to 8 \to 32 \to 128$, the fit visibly tightens, exactly what the
theorem promises. But notice: past a point, extra width brings diminishing returns and trains
*less* stably. **"Enough units in principle" is not the same as "easy to train in practice."**
This gap is what much of deep learning research is about.

### 4.2 Overfitting and the train/validation split

There is a second danger hiding in the wide-net fits. The targets carry noise, and a network with
enough capacity can drive the training loss down by bending to fit that noise rather than the
underlying $\sin(x)$. A fit that looks great on the points it was trained on can then be worse on
new inputs. This is **overfitting**, and you cannot detect it by watching the training loss alone:
a curve that wiggles through every noisy point has *low* training loss but poor generalization.

To see it, we need to measure error on data the model was not trained on. We split the dataset
into a **training set** (used to update parameters) and a **validation set** (held out, used only
to evaluate). Plotting both losses over training tells the two regimes apart: when training and
validation loss fall together the model is still learning real structure, but when training loss
keeps dropping while validation loss flattens or rises, the model has started memorizing noise.

### 🎯 Task 4.2 — Hold out a validation set

We have 200 noisy points. We set aside some of them as a **validation set** that the model never
trains on, keeping the rest for training. We shuffle before splitting so both sets span the whole
$x$-range: a contiguous split would put one arc of the sine in training and the rest in
validation, which would mix up overfitting with extrapolation to unseen regions. Use a small
training set (10 points) on purpose, since overfitting is easiest to see when the model has more
capacity than the data demands.

In [ ]:
torch.manual_seed(0)
perm = torch.randperm(x_reg.shape[0])   # shuffle so train/val both span the full x-range
n_train = 10
train_idx, val_idx = perm[:n_train], perm[n_train:]

# 🎯 TODO — index x_reg / y_reg with train_idx and val_idx
x_tr, y_tr = x_reg[train_idx], y_reg[train_idx]
x_val, y_val = x_reg[val_idx], y_reg[val_idx]
# end TODO

print(f"train points: {x_tr.shape[0]}   val points: {x_val.shape[0]}")

Next we need a training loop that watches both losses. It is the same five steps as the `train`
helper from section 3, with one addition: after each update we evaluate the loss on the
validation set under `torch.no_grad()`. That evaluation is read-only, so the validation data
influences neither the gradients nor the parameter updates. It only reports how the model performs
on points it was not fit to, which is exactly the signal that reveals overfitting.

In [ ]:
def train_with_val(model, X_tr, y_tr, X_val, y_val, loss_fn, epochs=4000, lr=0.01):
    """Same five-step loop as `train`, but also records validation loss each epoch.

    The validation set never enters backward() or step(): it is measured only,
    so it reports how the model does on data it was not fit to.
    """
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    tr_hist, val_hist = [], []
    for _ in tqdm(range(epochs)):
        optimizer.zero_grad()           # 1. clear gradients
        logits = model(X_tr)            # 2. forward on the TRAIN set
        loss = loss_fn(logits, y_tr)    # 3. training loss
        loss.backward()                 # 4. backward
        optimizer.step()                # 5. update
        tr_hist.append(loss.item())
        with torch.no_grad():           # validation: evaluate only, no grad, no update
            val_hist.append(loss_fn(model(X_val), y_val).item())
    return tr_hist, val_hist

Now train a deliberately oversized net (`H = 128`) on just the 10 training points and look at two
views side by side. On the left, the train and validation loss curves: watch for the moment they
separate, where training loss keeps falling but validation loss flattens or climbs. That gap is
overfitting. On the right, the learned function against the true $\sin(x)$: the network bending to
pass through individual noisy training points is the same phenomenon, seen in function space.

In [ ]:
torch.manual_seed(0)
net = make_net(128)   # high capacity: enough to chase the noise if nothing stops it
tr_hist, val_hist = train_with_val(net, x_tr, y_tr, x_val, y_val, mse, epochs=10000, lr=0.01)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# left: the two loss curves on a log scale, where the gap is easiest to read
axes[0].semilogy(tr_hist, label="train")
axes[0].semilogy(val_hist, label="validation")
axes[0].set_title("loss curves"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("MSE (log)")
axes[0].legend()

# right: the learned function against the true signal and the training points
xs = torch.linspace(-2 * np.pi, 2 * np.pi, 400).unsqueeze(1)
with torch.no_grad():
    ys = net(xs)
axes[1].scatter(x_tr.numpy(), y_tr.numpy(), s=12, color="#bbb", label="train data")
axes[1].plot(xs.numpy(), torch.sin(xs).numpy(), color="#2b2d6b", label="sin(x)")
axes[1].plot(xs.numpy(), ys.numpy(), color="#c0392b", label="net")
axes[1].set_title("learned fit"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"final train MSE: {tr_hist[-1]:.4f}")
print(f"final val MSE  : {val_hist[-1]:.4f}")

## 5. MLP churn classifier

> **Back to the customer.** In **Block 3** you used a classical sklearn model, logistic
> regression, to predict customer churn on the TELCO dataset. Today you replace it with a
> neural network and compare. The question is not "is the neural net better?", it is **when**
> the extra flexibility is worth its cost.

### 5.1 The TELCO churn dataset

The CSV at `data/customer_churn_TELCO.csv` holds **7 043 customer rows** from a fictional
telecom company. Each row describes one customer's account at a snapshot, with the target
`Churn ∈ {Yes, No}` recording whether they cancelled service.

The 19 features fall into four groups:

| Group | Columns |
|---|---|
| Demographics | `gender`, `SeniorCitizen`, `Partner`, `Dependents` |
| Account | `tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod` |
| Services | `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| Charges | `MonthlyCharges`, `TotalCharges` |

A few things to know before we touch the file:

- **`customerID`** is a pure identifier, the same lesson as Block 2's `Product ID`. Drop it.
- **`TotalCharges` has a string gotcha.** For brand-new customers (`tenure == 0`) the value is
  stored as a single space `" "`. `pd.to_numeric(..., errors="coerce")` turns those into NaNs;
  we then fill with `0.0`, which matches the business meaning ("no charges yet").
- The **positive rate is ≈ 26.5 %**, much less imbalanced than the Block 2 PdM dataset (3 %).
  Accuracy is no longer as misleading, but precision/recall/F1 still matter.

In [ ]:
DATA_PATH = DATA_PATH if "DATA_PATH" in globals() else "customer_churn_TELCO.csv"


def load_churn_data(path=DATA_PATH, test_size=0.2, seed=0):
    """Load the TELCO churn CSV and return (X_train, y_train, X_val, y_val, feature_names)."""
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    df = pd.read_csv(path)
    df = df.drop(columns=["customerID"])                       # pure identifier, no signal
    # "TotalCharges" has " " for tenure==0 customers -> coerce to NaN, then fill with 0
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
    df["Churn"] = (df["Churn"] == "Yes").astype(int)           # target to 0/1

    y = df.pop("Churn").to_numpy()
    X_df = pd.get_dummies(df, drop_first=True)                 # one-hot categoricals, drop_first avoids collinearity
    feature_names = list(X_df.columns)
    X = X_df.to_numpy(dtype=np.float32)

    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed,   # stratify keeps the ~26.5% rate in both splits
    )
    # scaler fit on TRAIN only, then applied to both: fitting on val would leak its statistics
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr).astype(np.float32)
    X_va = scaler.transform(X_va).astype(np.float32)

    to_t = lambda a, dt=torch.float32: torch.tensor(a, dtype=dt)
    return (
        to_t(X_tr), to_t(y_tr).unsqueeze(1).float(),           # targets shaped (N, 1) for BCE, as in the XOR section
        to_t(X_va), to_t(y_va).unsqueeze(1).float(),
        feature_names,
    )


X_train, y_train, X_val, y_val, feature_names = load_churn_data()
n_features = X_train.shape[1]
print("X_train:", X_train.shape, " positive rate:", y_train.mean().item())
print("X_val  :", X_val.shape,   " positive rate:", y_val.mean().item())
print("n_features:", n_features)
print("first 10 features:", feature_names[:10])

assert torch.isnan(X_train).sum() == 0, "no NaNs should remain after the TotalCharges fix"
assert X_train.shape[1] == X_val.shape[1], "train and val must have identical feature columns"
assert abs(y_train.mean().item() - y_val.mean().item()) < 0.02, "stratify should keep positive rates close"

### 5.2 A quick look at the data

Block 3 covered exploratory data analysis on a different (synthetic) dataset, so here we take just
a short look at the TELCO data itself before modeling. Two things are worth seeing: the class
balance (the churn rate we quoted as ≈ 26.5 %) and how a key numeric feature separates the two
classes. The overlap in `MonthlyCharges` between churners and non-churners is the kind of thing
that hints at whether a straight-line decision boundary will be enough, a question we return to
when we compare against the sklearn model.

In [ ]:
# short look at THIS dataset; Block 3 covers full EDA on its own (synthetic) data
df_raw = pd.read_csv(DATA_PATH)
df_raw["TotalCharges"] = pd.to_numeric(df_raw["TotalCharges"], errors="coerce").fillna(0.0)

print("Churn rate:")
print(df_raw["Churn"].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(8, 3.5))
for label, sub in df_raw.groupby("Churn"):
    ax.hist(sub["MonthlyCharges"], bins=30, alpha=0.55, label=f"Churn={label}")
ax.set_xlabel("MonthlyCharges"); ax.set_ylabel("count")
ax.set_title("monthly charges by churn status"); ax.legend(); plt.show()

### 5.3 TensorDataset and DataLoader

In sections 3 and 4 we did **full-batch** gradient descent: every parameter update used all of
the data at once. That is fine for four XOR points or 200 sine samples, but with thousands of
customer rows (and, in real problems, millions) we switch to **mini-batch** training, where each
update uses a small random subset of the data. PyTorch gives us two objects for this.

**`Dataset`** is an abstraction of an indexable collection of samples. It answers two questions:
how many samples are there (`len(dataset)`), and what is sample `i` (`dataset[i]`, returning a
`(features, target)` pair). Our data is already in tensors, so we use **`TensorDataset`**, which
simply wraps tensors and indexes them together row by row. For data that does not fit in memory or
needs decoding (images from disk, text from files), you would subclass `Dataset` yourself and
implement `__len__` and `__getitem__`, which is the same `__getitem__` protocol you saw for
classes in Block 2.

**`DataLoader`** wraps a `Dataset` and turns it into an iterable of **batches**. Each epoch it
optionally reshuffles the data, slices it into chunks of `batch_size`, and yields them one at a
time, so a training loop becomes a loop over batches inside a loop over epochs. It also handles
shuffling, dropping the last uneven batch, and parallel loading with worker processes, none of
which you have to write by hand.

**Why mini-batches, mathematically.** The true gradient is an average over the whole dataset,
$\nabla \mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} \nabla \ell_i$. A batch of size $B$ computes the
same average over a random subset:
$$\hat{g}_B = \frac{1}{B}\sum_{i \in \text{batch}} \nabla \ell_i.$$
Because the batch is sampled at random, $\hat{g}_B$ is an **unbiased estimator** of the true
gradient ($\mathbb{E}[\hat{g}_B] = \nabla \mathcal{L}$), with variance that shrinks as $1/B$. So
the batch size is a dial: small batches give cheap, noisy steps; large batches give expensive,
accurate ones. This single fact explains the whole setup:

1. **Cost.** Each step touches $B$ samples instead of $N$, so updates are far cheaper and we take
   many more of them per pass over the data. Datasets too large for memory can be streamed batch
   by batch.
2. **Useful noise.** The variance in $\hat{g}_B$ means each step points in a slightly wrong
   direction. That noise is not just tolerated, it helps: it can knock the optimizer out of sharp,
   shallow minima that a perfectly accurate full-batch step would settle into. This is what the
   "stochastic" in stochastic gradient descent refers to.
3. **Shuffling.** Reshuffling each epoch ensures successive batches are decorrelated, so the
   sequence of noisy gradients stays an unbiased walk toward the minimum rather than cycling
   through the data in a fixed, biased order.

Documentation: [`torch.utils.data.Dataset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset),
[`TensorDataset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset), and
[`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader). The
[data-loading overview](https://pytorch.org/docs/stable/data.html) ties the three together.

In [ ]:
train_ds = TensorDataset(X_train, y_train)   # wraps the tensors; train_ds[i] -> (features_i, target_i)
val_ds   = TensorDataset(X_val,   y_val)

# shuffle=True on train: reshuffle each epoch so batches stay decorrelated (see 5.3)
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)
# val: no shuffling needed (we only evaluate), larger batch since there is no backward pass
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)

### 🎯 Task 5.1 — Build the classifier

Architecture rationale:

- **Input:** `n_features` (the count printed when we loaded the data, after one-hot encoding the
  categoricals). Use the `n_features` variable rather than hard-coding a number.
- **Two hidden layers**, 32 then 16 units, with **ReLU** activation
  $$\phi(z) = \max(0, z).$$
  ReLU is the workhorse for tabular MLPs: cheap to compute, no saturation in the positive half so
  gradients stay alive (the saturation problem you saw with sigmoid and tanh in section 2.2), and
  empirically faster to train than $\tanh$.
- **Output:** a single logit. We use `nn.BCEWithLogitsLoss`, which applies the sigmoid internally
  and is more numerically stable than `BCELoss` plus a manual `sigmoid`, the same loss you used
  for XOR and parity.
- **Optimizer:** **Adam** at `lr=1e-3`. Intuition: SGD with **per-parameter adaptive learning
  rates** computed from running averages $m_t, v_t$ of the first and second moments of the
  gradient. You do not need the details today, Adam is a safe default for new architectures.

Build the `nn.Sequential` below.

In [ ]:
torch.manual_seed(0)

# 🎯 TODO — define the MLP: Linear(n_features, 32) -> ReLU -> Linear(32, 16) -> ReLU -> Linear(16, 1)
clf = nn.Sequential(
    nn.Linear(n_features, 32),   # input width comes from the data, not hard-coded
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),            # single logit: BCEWithLogitsLoss applies the sigmoid
)
# end TODO

clf_loss = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(clf.parameters(), lr=1e-3)   # Adam, so we cannot reuse the SGD train() helper
print(clf)

# self-check: shape of the first and last layers
assert isinstance(clf, nn.Sequential), "model should be an nn.Sequential"
assert clf[0].in_features == n_features, "input layer width must match the data"
assert clf[-1].out_features == 1, "output must be a single logit for binary BCE"

### 5.4 Training loop

This is the same five canonical steps as the `train` helper from section 3, with one structural
change: each epoch now loops over the **batches** that `train_loader` yields, taking one
optimizer step per batch instead of one per epoch. So there are two nested loops, epochs on the
outside and batches on the inside, and a single pass over all batches is one epoch. This is
mini-batch SGD made concrete: many cheap, slightly noisy steps per pass over the data, exactly the
tradeoff section 5.3 described.

Everything inside the batch loop is unchanged from before. `loss.backward()` still does the heavy
lifting: from the scalar batch loss, autograd walks the graph in reverse through every layer and
fills `.grad` for every parameter in one call. The only difference is that the loss is now
computed on a batch of rows rather than the full dataset, so each `.grad` is the unbiased,
slightly noisy gradient estimate from section 5.3.

In [ ]:
def epoch_loss(model, loader, loss_fn):
    """Average per-sample loss over a loader, with no_grad (evaluation only)."""
    total, n = 0.0, 0
    with torch.no_grad():                       # evaluation only: no graph, no .grad
        for xb, yb in loader:
            logits = model(xb)
            # weight each batch loss by its size, so the last (smaller) batch does not count equally
            total += loss_fn(logits, yb).item() * xb.size(0)
            n += xb.size(0)
    return total / n                            # true per-sample mean over the whole loader

### 🎯 Task 5.2 — Write the train step

Fill in the five lines that make up one gradient-descent step on a mini-batch. These are the same
five canonical steps from section 3, in the same order, now applied to one batch `(xb, yb)`:

1. `zero_grad` — clear the previous step's gradients (autograd accumulates, so this is not optional);
2. forward — pass `xb` through the model to get logits;
3. compute the loss against `yb`;
4. backward — `loss.backward()` lets autograd fill `.grad` for every parameter;
5. step — the optimizer reads those gradients and updates the weights.

After filling these in, the rest of the loop (iterating batches, recording losses) is already
written for you below.

In [ ]:
EPOCHS = 30
train_hist, val_hist = [], []
for ep in tqdm(range(EPOCHS)):                       # outer loop: epochs
    clf.train()                                # train mode (matters once a model has dropout/batchnorm)
    for xb, yb in train_loader:                # inner loop: mini-batches from the DataLoader
        # 🎯 TODO — one forward+backward+step on this minibatch (the five steps, canonical order)
        optimizer.zero_grad()    # 1. clear last step's gradients (autograd accumulates)
        logits = clf(xb)         # 2. forward on this batch
        loss = clf_loss(logits, yb)  # 3. batch loss
        loss.backward()          # 4. autograd fills .grad
        optimizer.step()         # 5. Adam updates the weights
        # end TODO
    clf.eval()                                 # eval mode for the loss measurements below
    train_hist.append(epoch_loss(clf, train_loader, clf_loss))
    val_hist.append(epoch_loss(clf, val_loader,   clf_loss))

plt.plot(train_hist, label="train")
plt.plot(val_hist,   label="val")
plt.title("training curves"); plt.xlabel("epoch"); plt.ylabel("BCE")
plt.legend(); plt.show()

# self-check: the model actually learned something
assert train_hist[-1] < train_hist[0], "final train loss should be below the initial loss"
print(f"final train BCE: {train_hist[-1]:.4f}   final val BCE: {val_hist[-1]:.4f}")

### 5.5 Evaluation

Block 2 hammered the point: **accuracy alone misleads on imbalanced data.** For churn, where
26.5 % of customers leave, the metrics that matter are:

$$\text{precision} = \frac{TP}{TP + FP}, \qquad
  \text{recall} = \frac{TP}{TP + FN}, \qquad
  F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}.$$

**Precision** answers *"of the customers we flagged, how many actually leave?"*, relevant when
acting on a flag costs money (sending retention offers).
**Recall** answers *"of the customers who leave, how many did we catch?"*, relevant when
*missing* a churner is costly.
$F_1$ is their harmonic mean, high only when both are high, which is why it is the single number
usually reported on imbalanced problems.

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

clf.eval()
with torch.no_grad():                          # inference only
    val_logits = clf(X_val)
    val_probs  = torch.sigmoid(val_logits)     # logits -> probabilities (the model never applies sigmoid itself)
    val_pred   = (val_probs > 0.5).int().squeeze().numpy()   # threshold at 0.5 for the predicted class
y_val_np = y_val.squeeze().int().numpy()

mlp_acc = accuracy_score(y_val_np, val_pred)
mlp_p, mlp_r, mlp_f1, _ = precision_recall_fscore_support(
    y_val_np, val_pred, average="binary", zero_division=0)

print("MLP accuracy        :", round(mlp_acc, 3))
print("MLP confusion matrix:\n", confusion_matrix(y_val_np, val_pred))
print(f"MLP precision: {mlp_p:.3f}   recall: {mlp_r:.3f}   f1: {mlp_f1:.3f}")

### 5.6 The "Block 3 model": sklearn baseline

Train the same classical model you built in Block 3, logistic regression, on the same processed
features and the same train/validation split, then print the two scoreboards side by side. Keeping
the data identical is what makes the comparison fair: any difference in the metrics comes from the
model, not from different preprocessing. Recall that logistic regression is a single linear layer
plus a sigmoid, exactly the model that failed on XOR in section 3, so this comparison asks whether
churn is closer to a linearly separable problem or one where the MLP's extra flexibility pays off.

In [ ]:
from sklearn.linear_model import LogisticRegression

X_tr_np = X_train.numpy()
y_tr_np = y_train.squeeze().numpy()
X_va_np = X_val.numpy()

logreg = LogisticRegression(max_iter=1000).fit(X_tr_np, y_tr_np)   # same features and split as the MLP
logreg_pred = logreg.predict(X_va_np)                              # sklearn thresholds at 0.5 internally
lr_acc = accuracy_score(y_val_np, logreg_pred)
lr_p, lr_r, lr_f1, _ = precision_recall_fscore_support(
    y_val_np, logreg_pred, average="binary", zero_division=0)

print("                     accuracy   precision   recall      f1")
print(f"sklearn LR        :   {lr_acc:.3f}      {lr_p:.3f}      {lr_r:.3f}      {lr_f1:.3f}")
print(f"PyTorch MLP (ours):   {mlp_acc:.3f}      {mlp_p:.3f}      {mlp_r:.3f}      {mlp_f1:.3f}")

**Closing thought.** On a tabular dataset of this size, a tuned linear model is hard to beat. The
extra flexibility of a neural network tends to earn its keep on data with rich structure (images,
sequences, text), where useful features must be learned, not on a few dozen one-hot dummies where
the relationship is already close to linear. The win of this block is **not** "neural beats
linear." It is that you now have the tooling, tensors, autograd, `nn.Module`, optimizers, and
`DataLoader`, to reach for a network the moment a problem actually needs one.

If the two scoreboards came out nearly tied, that is the intended result, not a failure of your
network. Knowing when *not* to add complexity is as much a part of the engineering judgment as
knowing how to build the model in the first place.

## 5.7 Coming up in Block 5

Today the goal was getting a network to train at all. Several things were deliberately hidden or
left half-explained, and the next block is where they get filled in. It moves from "getting a
network to train" to "training it correctly and efficiently."

Threads we left open today that Block 5 picks up:

- **`torch.no_grad()` and autograd context.** You wrapped every evaluation in `no_grad` and were
  told it means "inference only." Block 5 explains what that actually does (it stops PyTorch from
  building the computation graph), why that saves memory and compute, and how `requires_grad` and
  `inference_mode()` fit in.
- **`model.train()` vs `model.eval()`.** You called both without much explanation. Block 5 makes
  the distinction concrete: some layers (dropout, batch normalization) behave differently during
  training and inference, and these two calls are how you switch between the two regimes.
- **Where tensors live.** Everything today ran on CPU. Block 5 introduces **devices**: why a
  tensor or module lives on a specific device (CPU or GPU), how `.to(device)` moves it, and why
  every operand of an operation must be on the same device. You will move the churn MLP to a GPU
  and watch it speed up.

New machinery Block 5 adds on top:

- **Mixed precision** (`autocast` + `GradScaler`): storing activations in 16-bit roughly halves
  memory and raises throughput, but the narrow range of fp16 can underflow small gradients, which
  is what loss scaling exists to fix.
- **Checkpointing**: saving the `state_dict` of the model *and* the optimizer, so training can
  resume exactly where it stopped. You will see why the optimizer state matters, not just the
  weights.
- **Reproducibility**: seeding the Python, NumPy, and PyTorch RNGs and enabling deterministic
  algorithms, so two runs produce identical loss curves, along with what that costs in throughput
  and why an unseeded run is not wrong, only non-reproducible.
- **`einops`** (`rearrange`, `reduce`, `repeat`): tensor reshaping that names its axes explicitly
  and fails loudly on a shape mismatch, instead of the silent bugs that bare `view`/`reshape` can
  hide.

You will start from the exact churn MLP you just built and make it device-aware, then layer these
on one at a time.